In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                              recall_score, f1_score, confusion_matrix, classification_report)

base_path = '/Users/mohau/Desktop/STADIOalot-Capstone/datasets/'

In [3]:
# Load the engineered data
with open(base_path + 'X_train_engineered.pkl', 'rb') as f:
    X_train = pickle.load(f)
with open(base_path + 'X_test_engineered.pkl', 'rb') as f:
    X_test = pickle.load(f)
with open(base_path + 'y_train.pkl', 'rb') as f:
    y_train = pickle.load(f)
with open(base_path + 'y_test.pkl', 'rb') as f:
    y_test = pickle.load(f)

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train:", y_train.shape)
print("y_test: ", y_test.shape)

X_train: (800000, 27)
X_test:  (200000, 27)
y_train: (800000,)
y_test:  (200000,)


In [4]:
# HistGradientBoostingClassifier uses class_weight instead of scale_pos_weight
from sklearn.utils.class_weight import compute_sample_weight

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

print(f"Negative samples: {neg}")
print(f"Positive samples: {pos}")
print(f"Imbalance ratio: {neg/pos:.2f}:1")

# Compute sample weights to handle imbalance
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

Negative samples: 671825
Positive samples: 128175
Imbalance ratio: 5.24:1


In [5]:
model2 = HistGradientBoostingClassifier(
    max_iter=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    class_weight='balanced'
)

model2.fit(X_train, y_train)   # Note: sample_weights not needed, class_weight handles it
print("Model 2 trained successfully!")

Model 2 trained successfully!


In [6]:
y_pred = model2.predict(X_test)
y_prob = model2.predict_proba(X_test)[:, 1]
print(f"Predicted clicks: {y_pred.sum()}")

Predicted clicks: 86818


In [7]:
model_path = '/Users/mohau/Desktop/STADIOalot-Capstone/models/'

with open(model_path + 'model2_histgradient.pkl', 'wb') as f:
    pickle.dump(model2, f)

print("Model saved to:", model_path + 'model2_histgradient.pkl')

Model saved to: /Users/mohau/Desktop/STADIOalot-Capstone/models/model2_histgradient.pkl


In [8]:
print("MODEL 2: GRADIENT BOOSTING - RESULTS")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"AUC-ROC:   {roc_auc_score(y_test, y_prob):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

MODEL 2: GRADIENT BOOSTING - RESULTS
Accuracy:  0.6489
AUC-ROC:   0.7596
Precision: 0.2801
Recall:    0.7589
F1 Score:  0.4092

Confusion Matrix:
[[105455  62501]
 [  7727  24317]]

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.63      0.75    167956
           1       0.28      0.76      0.41     32044

    accuracy                           0.65    200000
   macro avg       0.61      0.69      0.58    200000
weighted avg       0.83      0.65      0.70    200000

